# 05 — Recovery, validation and training (audit 2026-11-24)
Run cells in order. **Stop at the first cell whose expected output you do not get.** After a runtime reset rerun Cell 1, then continue where you were.

In [ ]:
# CELL 1 — environment (rerun after every reset)
import os, sys, subprocess, torch
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'): drive.mount('/content/drive')
REPO  = '/content/HMNet_SETG'                         # code
DATA  = '/content/qvhighlights'                       # local copy of the dataset (feature root)
NLVMS = '/content/drive/MyDrive/nlvms'                # checkpoints / logs / locks live on Drive
DRIVE_DATA = '/content/drive/MyDrive/qvhighlights'    # <-- EDIT: where the dataset lives on Drive
os.makedirs(NLVMS, exist_ok=True); os.chdir(REPO); sys.path.insert(0, REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'omegaconf', 'h5py', 'scipy', 'scikit-learn', 'spacy'], check=True)
import spacy
try: spacy.load('en_core_web_sm')
except OSError: subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm', '-q'], check=True)
from src.models.mamba_block import MAMBA_SSM_AVAILABLE
print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('mamba_ssm installed:', MAMBA_SSM_AVAILABLE, '| DRIVE_DATA exists:', os.path.exists(DRIVE_DATA))

In [ ]:
# CELL 2 — dataset facts (read-only; uses only numpy, no project code)
import json, random, collections, numpy as np
from pathlib import Path
D = Path(DATA)
rows = {s: [json.loads(l) for l in open(D/f'{s}.jsonl') if l.strip()] for s in ('train', 'val')}
r = rows['train'][0]
print('text_feature =', r['text_feature'], '| len(token_ids) =', len(r['token_ids']), '| saliency rows =', len(r['saliency']))
for s, R in rows.items():
    mt = sum(not (D/x['text_feature']['path']).exists() for x in R)
    mv = sum(not (D/v['path']).exists() for x in R for v in x['video_features'])
    print(f'{s}: {len(R)} rows | missing text files {mt} | missing video files {mv}')
print('qid*.npz in text/clip_text_features:', len(list((D/'text/clip_text_features').glob('qid*.npz'))))
random.seed(0)
have = [x for x in rows['train'] if (D/x['text_feature']['path']).exists()]
samp = random.sample(have, min(200, len(have)))
txt, vid = collections.Counter(), collections.Counter()
for x in samp:
    z = np.load(D/x['text_feature']['path']); t = z[x['text_feature']['key']]
    txt[f"keys={sorted(z.files)} dim={t.shape[1]} rows-len(token_ids)={t.shape[0]-len(x['token_ids'])}"] += 1
    (a, da), (b, db) = [np.load(D/v['path'])[v['key']].shape for v in x['video_features']]
    vid[f'SlowFast-CLIP rows={a-b} dims=({da},{db}) CLIP rows-saliency rows={b-len(x["saliency"])}'] += 1
print('text sample :', dict(txt)); print('video sample:', dict(vid))

In [ ]:
# CELL 3 — install the audited fixes (backs up your current code to Drive first)
import zipfile, hashlib, shutil, time, json
PATCH = f'{NLVMS}/HMNet_SETG_patch_2026-11-24.zip'      # upload the patch ZIP here first
backup = f'{NLVMS}/HMNet_SETG_code_backup_{time.strftime("%Y%m%d_%H%M")}'
shutil.copytree(REPO, backup, ignore=shutil.ignore_patterns('__pycache__', '*.pt', '.pytest_cache'))
z = zipfile.ZipFile(PATCH); info = json.loads(z.read('HMNet_SETG/PATCH_INFO.json'))
changed_locally = [f for f, h in info['replaces'].items() if hashlib.sha256(open(f'{REPO}/{f}', 'rb').read()).hexdigest() != h]
print('backup:', backup)
print('files you had edited yourself (will be replaced; your version is in the backup):', changed_locally or 'none')
z.extractall('/content')          # ZIP root is HMNet_SETG/ -> overwrites only the patched files
!python -m pytest -q 2>&1 | tail -2

In [ ]:
# CELL 4 — text-feature audit (read-only) + validation of both manifests
!python scripts/audit_text_features.py --feature-root {DATA} --manifests {DATA}/train.jsonl {DATA}/val.jsonl --search-dir "{DRIVE_DATA}" --output outputs/text_audit.json
for s in ('train', 'val'):
    !python scripts/validate_dataset.py --manifest {DATA}/{s}.jsonl --feature-root {DATA} --output outputs/validation_{s}.json

### Cell 4R — only if Cell 4 reports missing text files
Option A restores them from your Drive copy. Option B extracts **only** the text folder from the official tarball. Neither overwrites existing files. After A or B, rerun Cell 4.

In [ ]:
# CELL 4R-A — restore missing text files from the Drive copy (no overwrite), then save them back to Drive too
!python scripts/audit_text_features.py --feature-root {DATA} --manifests {DATA}/train.jsonl {DATA}/val.jsonl --search-dir "{DRIVE_DATA}" --copy-missing-from --output outputs/text_audit_restore.json
!mkdir -p "{DRIVE_DATA}/text/clip_text_features" && cp -n {DATA}/text/clip_text_features/qid*.npz "{DRIVE_DATA}/text/clip_text_features/"

In [ ]:
# CELL 4R-B — only if 4R-A still reports missing files: extract ONLY clip_text_features from the official tarball
TAR = '/content/drive/MyDrive/moment_detr_features.tar.gz'   # <-- EDIT: your copy of the official tarball
!tar -tzf "{TAR}" | grep -m 2 "clip_text_features/qid"
!mkdir -p /content/tar_text && tar -xzf "{TAR}" -C /content/tar_text --wildcards "*clip_text_features/qid*"
!python scripts/audit_text_features.py --feature-root {DATA} --manifests {DATA}/train.jsonl {DATA}/val.jsonl --search-dir /content/tar_text --copy-missing-from --output outputs/text_audit_tar.json
!cp -n {DATA}/text/clip_text_features/qid*.npz "{DRIVE_DATA}/text/clip_text_features/"

In [ ]:
# CELL 5 — load one real training sample and one real validation sample
from src.utils.config import load_config
from src.data.dataset import FeatureMomentDataset
cfg_b = load_config('configs/qvhighlights.yaml'); cfg_s = load_config('configs/qvhighlights_setg.yaml')
assert cfg_b.model.use_setg is False and cfg_s.model.use_setg is True
train_ds = FeatureMomentDataset(f'{DATA}/train.jsonl', DATA, cfg_b.data.max_subevents)
val_ds = FeatureMomentDataset(f'{DATA}/val.jsonl', DATA, cfg_b.data.max_subevents)
for name, ds in (('train', train_ds), ('val', val_ds)):
    x = ds[0]
    print(name, x['qid'], '| video', tuple(x['video'].shape), '| text', tuple(x['text'].shape), '| windows', x['spans'].tolist(),
          '| K =', x['num_subevents'], '| parser:', x['parse']['status'])

In [ ]:
# CELL 6 — one real training batch
from torch.utils.data import DataLoader
from src.data.collate import collate_moments
BATCH = 32
loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, collate_fn=lambda b: collate_moments(b, 4))
batch = next(iter(loader))
print('video', tuple(batch['video'].shape), '| text', tuple(batch['text'].shape),
      '| valid clips per video', batch['video_mask'].sum(1)[:6].tolist(), '| K per query', batch['subevent_mask'].sum(1)[:6].tolist(),
      '| saliency', None if batch['saliency'] is None else tuple(batch['saliency'].shape))

In [ ]:
# CELL 7 — forward pass on GPU (baseline and SE-TG built exactly as scripts/train.py builds them)
import torch, time
from src.models.hmnet import HMNet
dev = torch.device('cuda')
def build(cfg):
    return HMNet(video_dim=cfg.data.video_dim, text_dim=cfg.data.text_dim, hidden_dim=cfg.model.hidden_dim, heads=cfg.model.heads,
                 state_dim=cfg.model.state_dim, enhancement_layers=cfg.model.enhancement_layers, cofusion_layers=cfg.model.cofusion_layers,
                 num_queries=cfg.model.num_queries, decoder_layers=cfg.model.decoder_layers, max_seq_len=cfg.model.max_seq_len,
                 dropout=cfg.model.dropout, drop_path=cfg.model.drop_path, use_setg=cfg.model.use_setg, max_subevents=cfg.data.max_subevents,
                 use_reconstruction=cfg.model.use_reconstruction, vocab_size=cfg.data.vocab_size,
                 reconstruction_mask_fraction=cfg.reconstruction.mask_fraction, reconstruction_temperature=cfg.reconstruction.temperature,
                 reconstruction_positive_iou=cfg.reconstruction.positive_iou).to(dev)
def inputs(b):
    return dict(video=b['video'].to(dev), text=b['text'].to(dev), video_mask=b['video_mask'].to(dev), text_mask=b['text_mask'].to(dev),
                subevent_spans=b['token_spans'].to(dev), subevent_mask=b['subevent_mask'].to(dev),
                token_ids=None if b['token_ids'] is None else b['token_ids'].to(dev), gt_spans=[g.to(dev) for g in b['gt_spans']],
                video_ids=b['video_ids'], token_loss_mask=b['mlm_mask'].to(dev))
models = {}
for name, cfg in (('baseline', cfg_b), ('setg', cfg_s)):
    torch.manual_seed(0); m = build(cfg).train(); models[name] = m
    with torch.autocast('cuda', dtype=torch.float16):
        out = m(**inputs(batch))
    print(name, '| pred_spans', tuple(out['pred_spans'].shape), '| pred_logits', tuple(out['pred_logits'].shape),
          '| SE-TG outputs' if 'subevent_alignments' in out else '| no SE-TG outputs',
          '| params %.1fM' % (sum(p.numel() for p in m.parameters()) / 1e6))

In [ ]:
# CELL 8 — loss + backward + optimizer step, with time and memory per iteration
from src.losses.total_loss import compute_total_loss
for name, cfg in (('baseline', cfg_b), ('setg', cfg_s)):
    m = models[name]; opt = torch.optim.AdamW(m.parameters(), lr=2e-4); scaler = torch.amp.GradScaler('cuda')
    torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize(); t = time.time()
    with torch.autocast('cuda', dtype=torch.float16):
        out = m(**inputs(batch))
        loss, parts = compute_total_loss(out, batch['targets'], [g.to(dev) for g in batch['gt_spans']], batch['video_mask'].to(dev),
                                         saliency_scores=batch['saliency'].to(dev), video_ids=batch['video_ids'], **dict(cfg.loss))
    scaler.scale(loss).backward(); scaler.unscale_(opt)
    finite = all(torch.isfinite(p.grad).all() for p in m.parameters() if p.grad is not None)
    scaler.step(opt); scaler.update(); opt.zero_grad(); torch.cuda.synchronize()
    print(f"{name}: loss {loss.item():.3f} finite_grads={finite} | {time.time()-t:.2f}s/iter | peak {torch.cuda.max_memory_allocated()/2**30:.1f} GB")
    print('   ', {k: round(float(v), 3) for k, v in parts.items()})
del models; torch.cuda.empty_cache()

In [ ]:
# CELL 9 — smoke training: 256 train / 128 val rows, 1 epoch, local disk only (nothing written to Drive)
!rm -rf /content/smoke_runs && head -n 256 {DATA}/train.jsonl > /content/smoke_train.jsonl && head -n 128 {DATA}/val.jsonl > /content/smoke_val.jsonl
import time; t = time.time()
!python scripts/train.py --config configs/qvhighlights.yaml --manifest /content/smoke_train.jsonl --val-manifest /content/smoke_val.jsonl --feature-root {DATA} --run-name smoke_base --fresh --set train.epochs=1 train.batch_size=32 train.grad_accum=1 paths.drive_root=/content/smoke_runs 2>&1 | tail -1 | cut -c1-400
sec = time.time() - t
print(f'{sec:.0f}s for 256+128 rows -> rough full-epoch estimate {sec*(7218+1550)/384/60:.0f} min (includes start-up; measure again in Cell 10)')

### Cell 10 — full training (baseline first, then SE-TG)
Run **one** of the two lines per session. To resume after a disconnect: rerun Cell 1, then **exactly the same line** (same run name, same `--set` values). The run refuses to resume if the config, manifests or Mamba backend changed.

In [ ]:
# CELL 10 — full training. Checkpoints: {NLVMS}/checkpoints/<run-name>/ (epoch_XXX.pt, best.pt, experiment_manifest.json)
COMMON = f"--manifest {DATA}/train.jsonl --val-manifest {DATA}/val.jsonl --feature-root {DATA} --set train.batch_size=32 train.grad_accum=1 paths.drive_root={NLVMS}"
!python scripts/train.py --config configs/qvhighlights.yaml      --run-name qvh_hmnet_baseline_v2 {COMMON}
# !python scripts/train.py --config configs/qvhighlights_setg.yaml --run-name qvh_hmnet_setg_v2     {COMMON}

In [ ]:
# CELL 11 — evaluation with the official QVHighlights evaluator (validation split)
import json
!test -d /content/moment_detr || git clone -q https://github.com/jayleicn/moment_detr /content/moment_detr
for cfg, run in (('configs/qvhighlights.yaml', 'qvh_hmnet_baseline_v2'), ('configs/qvhighlights_setg.yaml', 'qvh_hmnet_setg_v2')):
    out = f'{NLVMS}/logs/eval_{run}'
    !python scripts/evaluate.py --config {cfg} --manifest {DATA}/val.jsonl --feature-root {DATA} --checkpoint {NLVMS}/checkpoints/{run}/best.pt --output-dir {out}
    !cd /content/moment_detr && PYTHONPATH=. python standalone_eval/eval.py --submission_path {out}/predictions_official.jsonl --gt_path {out}/gt_official.jsonl --save_path {out}/metrics_official.json --not_verbose
    print(run, json.load(open(f'{out}/metrics_official.json'))['brief'])